# NetSciX 2027 · Notebook 1 — Spatial prediction

**Case study:** Cañete Valley, Peru · **candidate common support:** historical 250 m grid, 12,826 cells (~801.625 km²) · CRS: EPSG:32718.

**Teaching question:** Can environmental covariates distinguish cells *containing a previously recorded point* from **unlabelled background** under spatial cross-validation?

⚠️ **Important:** The label `registered_point_cell = 0` means **no prior point was recorded in this cell**, *not* an archaeological absence. Scores quantify discrimination between registered-positive cells and unlabelled background, **not archaeological site-detection accuracy**. The candidate domain is not yet a final approved scientific study area.

The **real dataset is private** because the combination of 250 m coordinates and site-registration labels may expose sensitive heritage locations. Do **not** upload real data, resulting cell-level probabilities, or marked maps to a public repository. The public notebook contains **no original archaeological coordinates**. It can also run using clearly labelled **synthetic demo data**.

### 0. Data access (Google Colab)

- For private analysis, select `DATA_MODE="real"` and upload `canete_250m_PRIVATE.csv` when prompted (from the **separate private ZIP**, not the public GitHub repository).
- Default `DATA_MODE="demo"`: generate a synthetic dataset *without any archaeological coordinates*. This mode is for public demonstrations and testing, not scientific results.
- For noninteractive local testing, set environment variable `NETSCIX_PRIVATE_DATA=/absolute/path/to/canete_250m_PRIVATE.csv`.

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold
from sklearn.metrics import average_precision_score, roc_auc_score

DATA_MODE = os.getenv("NETSCIX_DATA_MODE", "demo")  # "real" | "demo"
EXPORT_OOF = False  # Set True only to save a private handoff for Notebook 2.
RANDOM_STATE = 2027
FEATURES = ["altitud_m", "pendiente_grados", "dist_acequia_m_aprox", "dist_rio_m_aprox"]
print("Data mode:", DATA_MODE)

In [ ]:
def make_synthetic_demo(seed=2027):
    rng = np.random.default_rng(seed)
    xx, yy = np.meshgrid(np.arange(30), np.arange(26))
    x, y = xx.ravel().astype(float) * 250, yy.ravel().astype(float) * 250
    z = 20 + .003*x + .001*y + 10*np.sin(x/1500)
    slope = np.clip(4 + 3*np.cos(y/900) + rng.normal(0,1.2,len(x)),0,None)
    river = np.abs(y - (3200 + 550*np.sin(x/1100)))
    canal = np.abs(x - 3400)
    score = -.04*z - .2*slope - .0015*river + .0003*canal
    score = (score-score.mean())/(score.std()+1e-12)
    prob = .012 + .12/(1+np.exp(-score))
    label = rng.binomial(1,prob).astype(int)
    return pd.DataFrame(dict(cell_id=[f"SIM_{i:04d}" for i in range(len(x))],
        x=x,y=y,registered_point_cell=label,
        background_status="SYNTHETIC_UNLABELLED",
        altitud_m=z,pendiente_grados=slope,
        dist_acequia_m_aprox=canal,dist_rio_m_aprox=river))

if DATA_MODE == "demo":
    df = make_synthetic_demo()
    print("SYNTHETIC demonstration — no empirical archaeological records.")
elif DATA_MODE == "real":
    manual = os.getenv("NETSCIX_PRIVATE_DATA", "")
    candidates = ([Path(manual)] if manual else []) + [Path("/content/canete_250m_PRIVATE.csv"), Path("data/private/canete_250m_PRIVATE.csv"), Path("../data/private/canete_250m_PRIVATE.csv")]
    csv_file = next((p for p in candidates if p.is_file()), None)
    if csv_file is None:
        try:
            from google.colab import files
        except ImportError as e:
            raise FileNotFoundError("Set NETSCIX_PRIVATE_DATA to the private CSV path or switch DATA_MODE='demo'.") from e
        print("Upload PRIVATE canete_250m_PRIVATE.csv (never publish it on GitHub).")
        uploaded = files.upload()
        if "canete_250m_PRIVATE.csv" not in uploaded:
            raise FileNotFoundError("Required private CSV was not uploaded")
        csv_file = Path("canete_250m_PRIVATE.csv")
    df = pd.read_csv(csv_file)
    print("PRIVATE real Cañete data loaded. Do not share site-labelled cell-level outputs.")
else:
    raise ValueError("DATA_MODE must be 'real' or 'demo'")

required = FEATURES+["cell_id", "x", "y", "registered_point_cell", "background_status"]
assert set(required).issubset(df.columns)
assert not df.duplicated(["cell_id"]).any()
assert set(df.registered_point_cell.dropna().unique()).issubset({0,1})
print(f"Grid cells: {len(df):,}; registered-positive cells: {df.registered_point_cell.sum():,}; unlabelled background: {(df.registered_point_cell==0).sum():,}")
print("Missing values by predictor:", df[FEATURES].isna().sum().to_dict())

### 1. Spatial grouping

Create 3 km spatial blocks using projected UTM coordinates. `GroupKFold` keeps all cells of a block entirely in train **or** validation, reducing random-cell spatial leakage. This remains a **pedagogical spatial validation**: proximity across block boundaries and source-selection bias are still possible. The split does not establish survey-confirmed negatives.

In [ ]:
block_m = 3000
bx = np.floor((df["x"]-df["x"].min())/block_m).astype(int)
by = np.floor((df["y"]-df["y"].min())/block_m).astype(int)
groups = bx.astype(str)+":"+by.astype(str)
y = df["registered_point_cell"].astype(int)
print(f"Distinct spatial blocks: {groups.nunique()}")
splitter = GroupKFold(n_splits=5)
folds = list(splitter.split(df[FEATURES], y, groups))
for i, (train, test) in enumerate(folds, 1):
    train_groups = set(groups.iloc[train]); test_groups = set(groups.iloc[test]); assert train_groups.isdisjoint(test_groups)
    print(f"Fold {i}: train={len(train):5d}, test={len(test):5d}, registered+ test={int(y.iloc[test].sum()):3d}")

### 2. Random Forest: register-positive vs unlabelled background

Features are **elevation**, **slope**, **approximate distance to irrigation channels**, and **approximate distance to rivers**. We **exclude** coordinates and distances to archaeological sites as predictors to avoid directly encoding label definition. Hydrological distances come from approximate rasterized lines, not historically time-specific water systems.

Missing slopes are imputed **inside each training fold**. Class weighting addresses imbalance computationally, but does **not** turn unlabelled background into true negatives.

In [ ]:
def build_pipeline():
    return Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("rf", RandomForestClassifier(n_estimators=160,min_samples_leaf=6,
           max_features="sqrt",class_weight="balanced_subsample",
           n_jobs=-1,random_state=RANDOM_STATE))
    ])

out_of_fold = np.full(len(df),np.nan)
records = []
for fold_id, (train,test) in enumerate(folds,1):
    model = build_pipeline()
    model.fit(df.iloc[train][FEATURES],y.iloc[train])
    score = model.predict_proba(df.iloc[test][FEATURES])[:,1]
    out_of_fold[test] = score
    yt = y.iloc[test]
    base = float(yt.mean())
    ap = average_precision_score(yt,score)
    auc = roc_auc_score(yt,score)
    records.append({"fold":fold_id,"registered_positive":int(yt.sum()),
        "cells":len(test),"background_reference_rate":base,
        "AP_recorded_vs_background":ap,
        "ROC_AUC_recorded_vs_background":auc,
        "AP_divided_by_background_rate":ap/base if base>0 else np.nan})
metrics = pd.DataFrame(records)
assert np.isfinite(out_of_fold).all()
print(metrics.round(4).to_string(index=False))
print("Mean AP (fold mean):",round(metrics.AP_recorded_vs_background.mean(),4))

### 3. Compare ranking to the recorded-background baseline

**Average precision (AP)** summarizes ranking of **previously registered cells versus unlabelled cells**. Compare AP to the registered-cell frequency in each test fold. These values are *conditional on the incomplete inventory and spatial block design*; they do not measure archaeological detection performance or generalize to newly surveyed sites without independent field evaluation.

In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
ax.plot(metrics.fold,metrics.AP_recorded_vs_background,'o-',label="Recorded-vs-background AP")
ax.plot(metrics.fold,metrics.background_reference_rate,'s--',label="Recorded frequency (baseline)")
ax.set(xlabel="Spatial CV fold",ylabel="Score",title="Spatially blocked evaluation (interpretation limited)")
ax.set_xticks(metrics.fold)
ax.legend(); plt.show()

### 4. Candidate scores and exploratory map

Fit a model on the full grid only to **illustrate the spatial scoring workflow**. Its output is *in-sample* and **must not be reported as independent predictive performance**. The out-of-fold predictions above support the spatial validation diagnostic. Never share a real-cell score map publicly without heritage-data review.

In [ ]:
final_model = build_pipeline()
final_model.fit(df[FEATURES],y)
df["in_sample_exploratory_score"] = final_model.predict_proba(df[FEATURES])[:,1]
fig,ax=plt.subplots(figsize=(8,5))
sc=ax.scatter(df["x"],df["y"],c=df["in_sample_exploratory_score"],s=4,cmap="viridis",rasterized=True)
ax.set(xlabel="UTM E (m)",ylabel="UTM N (m)",title=f"{'SYNTHETIC' if DATA_MODE=='demo' else 'PRIVATE — DO NOT PUBLISH'} exploratory spatial scores")
fig.colorbar(sc,ax=ax,label="RF score (in-sample; not validated probability)")
plt.show()

### 5. Optional private export

To move on to **Notebook 2**, a vetted selection policy is needed to choose candidate nodes from scores. No site-labelled cell-level data or probability maps are exported automatically; that link is postponed until the team explicitly approves how to handle heritage-location information.

**Discussion:** How do block size, a different background sample, or uncertainty about site geometries affect the meaning of these rankings? Can held-out *independent* field data test their usefulness?


## 5. Optional out-of-fold handoff to Notebook 2

The variable `out_of_fold` contains exactly one prediction for each cell, obtained from a spatial fold in which that cell was excluded from training. These scores support a **ranked candidate-selection exercise** in Notebook 2; they are **not site-presence probabilities**. Change `EXPORT_OOF=True` in the initial configuration and rerun this last cell to create a local CSV.

In `real` mode the exported CSV is **sensitive archaeological-derived information**. Keep it only in a private folder and never commit it to the public repository. In `demo` mode the exported records are synthetic.


In [ ]:

if EXPORT_OOF:
    tag = 'PRIVATE' if DATA_MODE == 'real' else 'SYNTHETIC'
    destination = Path('/content') if Path('/content').exists() else Path.cwd()
    exported = pd.DataFrame({'cell_id': df['cell_id'], 'oof_rank_score': out_of_fold})
    assert exported.cell_id.is_unique and exported.oof_rank_score.notna().all()
    path = destination / f'nb1_oof_scores_{tag}.csv'
    exported.to_csv(path, index=False)
    print('Saved:', path, '|', 'RESTRICTED: DO NOT PUBLISH' if DATA_MODE == 'real' else 'synthetic demonstration')
else:
    print('OOF export disabled. Use EXPORT_OOF=True only if an explicit file is needed.')


**Try it:** Change `block_m` from 3000 to 2000 or 4000 in section 1 and rerun the notebook. How stable is average precision relative to the recorded-cell baseline? Explain why none of these metrics establish a verified absence class.